# L07-02｜Qwen3-0.6B 的 LoRA smoke test


## 实验目标

本节用 3 个训练 step 跑通一条最小 LoRA SFT 流程：读取对话数据、加载 Qwen3-0.6B、在 Ascend NPU 上完成前向与反向、保存适配器 checkpoint，并把 loss 写入日志。一个 step 完成一次参数更新。

本节沿用 L07-01 的 LoRA 公式 `W' = W + (alpha / r) × B × A`。基础模型权重保持冻结，训练器只更新 `A`、`B`。3 个 step 用于确认训练链路可用，正式的 loss 变化在 L07-03 观察。


## 实验流程

| 阶段 | 代码 | 观察重点 |
| --- | --- | --- |
| 数据 | 生成并读取 JSONL | `messages` 是否包含角色和内容 |
| LoRA | 组装 `swift sft` 命令 | `tuner_type`、`rank`、`alpha` 和目标层 |
| 训练 | 在单卡 NPU 上更新 3 次 | `global_step` 是否到达 3 |
| 输出 | 读取 `logging.jsonl` | loss 记录和 checkpoint 是否生成 |


## 1. 初始化环境、模型和数据

代码会安装缺失的 Python 包，加载 CANN 环境，检查 `torch_npu`，并从 ModelScope 获取 `Qwen/Qwen3-0.6B`。训练数据由代码写入 `WORK_DIR/train.jsonl`，每行是一条 `messages` 对话记录。

运行前确认 JupyterLab 使用 `Python (PyTorch-2.7.1)` kernel；这个 kernel 与 ModelArts 镜像中的 PyTorch、torch_npu 和 CANN 版本配套。

In [ ]:
from __future__ import annotations

import importlib.util
import json
import math
import os
import shlex
import shutil
import site
import subprocess
import sys
from pathlib import Path

def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)

require((3, 10) <= sys.version_info[:2] <= (3, 12), '本流程需要 Python 3.10、3.11 或 3.12；请使用带匹配 Ascend 运行时的 ModelArts 镜像。')
def install_missing_packages() -> None:
    packages = {'modelscope': 'modelscope', 'swift': 'ms-swift', 'matplotlib': 'matplotlib'}
    missing = [dist for module, dist in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        print('安装缺失依赖：', missing)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', *missing])
    python_bin = str(Path(sys.executable).parent)
    user_bin = str(Path(site.getuserbase()) / 'bin')
    old_path = os.environ.get('PATH', '')
    path_entries = old_path.split(os.pathsep) if old_path else []
    for candidate in (python_bin, user_bin):
        if candidate not in path_entries:
            old_path = candidate + os.pathsep + old_path
            path_entries.insert(0, candidate)
    os.environ['PATH'] = old_path

install_missing_packages()

def load_ascend_env() -> None:
    candidates = [
        Path('/usr/local/Ascend/ascend-toolkit/set_env.sh'),
        Path('/usr/local/Ascend/ascend-toolkit/latest/set_env.sh'),
    ]
    ascend_root = Path('/usr/local/Ascend')
    if ascend_root.is_dir():
        candidates.extend(sorted(ascend_root.glob('**/set_env.sh')))
    seen = set()
    for script in candidates:
        if not script.is_file() or str(script) in seen:
            continue
        seen.add(str(script))
        result = subprocess.run(
            ['bash', '-lc', f'source {shlex.quote(str(script))} >/dev/null 2>&1 && env -0'],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False,
        )
        if result.returncode != 0:
            continue
        for item in result.stdout.split(b'\0'):
            if b'=' in item:
                key, value = item.split(b'=', 1)
                os.environ[key.decode()] = value.decode(errors='ignore')
        print('已加载 Ascend 环境：', script)
        return
    print('未找到 CANN set_env.sh；继续使用当前 ModelArts 进程环境。')

load_ascend_env()
os.environ.setdefault('ASCEND_RT_VISIBLE_DEVICES', '0')
import torch
try:
    import torch_npu  # noqa: F401
except Exception as exc:
    raise RuntimeError('当前环境无法导入 torch_npu。请使用带 Ascend/CANN 运行时的 ModelArts 镜像。') from exc
require(hasattr(torch, 'npu'), '当前 PyTorch 没有 torch.npu；请检查 ModelArts 的 Ascend 运行时。')
npu_count = torch.npu.device_count()
require(npu_count > 0, '没有检测到 NPU。请确认 ModelArts 实例规格和可见设备。')
torch_version = getattr(torch, '__version__', 'unknown')
torch_npu_version = getattr(torch_npu, '__version__', 'unknown')
def major_minor(version: str) -> tuple[int, int] | None:
    try:
        parts = version.split('+', 1)[0].split('.')
        return int(parts[0]), int(parts[1])
    except (IndexError, ValueError):
        return None
if major_minor(torch_version) and major_minor(torch_npu_version):
    require(major_minor(torch_version) == major_minor(torch_npu_version), f'torch 与 torch_npu 版本不匹配：{torch_version} vs {torch_npu_version}。请按同一套 CANN/PyTorch/torch_npu 重新准备 ModelArts 镜像。')
torch.npu.set_device(0)
probe = torch.zeros(1, device='npu:0')
del probe

NOTEBOOK_ID = 'L07-02'
WORK_DIR = Path(os.environ.get('L07_WORK_DIR', str(Path.cwd() / f'{NOTEBOOK_ID}_workspace'))).expanduser()
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = os.environ.get('L07_MODEL_ID', 'Qwen/Qwen3-0.6B')
MODEL_CACHE = Path(os.environ.get('MODELSCOPE_CACHE', str(WORK_DIR / 'modelscope_cache'))).expanduser()
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
MODEL_PATH_OVERRIDE = os.environ.get('L07_MODEL_PATH')
if MODEL_PATH_OVERRIDE:
    MODEL_PATH = Path(MODEL_PATH_OVERRIDE).expanduser()
    require(MODEL_PATH.is_dir(), f'指定的模型目录不存在：{MODEL_PATH}')
else:
    from modelscope import snapshot_download
    MODEL_PATH = Path(snapshot_download(MODEL_ID, cache_dir=str(MODEL_CACHE)))
require((MODEL_PATH / 'config.json').is_file(), f'模型目录缺少 config.json：{MODEL_PATH}')
MODEL_REF = str(MODEL_PATH)

def qwen3_answer(text: str) -> str:
    return '<think>\n\n</think>\n\n' + text

examples = [
    ('请用一句话解释 LoRA 与全参数微调的区别。', 'LoRA 只训练注入的低秩适配器参数，全参数微调会更新模型的全部参数。'),
    ('lora_rank 控制什么？', '它是低秩分支的维度，决定适配器的参数量和表达容量。'),
    ('lora_alpha 控制什么？', '它控制低秩更新的缩放，不等同于 learning_rate。'),
    ('target_modules=all-linear 表示什么？', '它表示向模型中的线性层注入 LoRA 适配器。'),
    ('一个 batch 有 2 条样本，梯度累积 4 步时有效 batch size 是多少？', '单卡且不丢弃样本时，有效 batch size 是 2 乘以 4，也就是 8。'),
    ('SFT 数据中的 assistant 消息有什么作用？', '它提供模型需要学习的目标答案。'),
    ]
records = [
    {'messages': [
        {'role': 'system', 'content': '你是一个简洁、准确的课程实验助手。'},
        {'role': 'user', 'content': question + ' /no_think'},
        {'role': 'assistant', 'content': qwen3_answer(answer)},
    ]}
    for question, answer in examples
]
TRAIN_DATA = WORK_DIR / 'train.jsonl'
with TRAIN_DATA.open('w', encoding='utf-8') as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
loaded_records = [json.loads(line) for line in TRAIN_DATA.read_text(encoding='utf-8').splitlines() if line.strip()]
require(len(loaded_records) == len(records), '生成的 JSONL 条数不一致。')
require(all('messages' in item for item in loaded_records), '每条训练数据都必须包含 messages。')
environment_record = {'python': sys.version.split()[0], 'torch': getattr(torch, '__version__', 'unknown'), 'torch_npu': getattr(torch_npu, '__version__', 'unknown'), 'npu_count': npu_count, 'visible_npus': os.environ['ASCEND_RT_VISIBLE_DEVICES'], 'model_id': MODEL_ID, 'model_path': str(MODEL_PATH), 'train_data': str(TRAIN_DATA)}
(WORK_DIR / 'environment.json').write_text(json.dumps(environment_record, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

SWIFT_BIN = shutil.which('swift')
require(SWIFT_BIN is not None, '找不到 swift 命令；请确认 ms-swift 已安装且当前 Python 的 bin 目录在 PATH 中。')
VISIBLE_NPUS = os.environ['ASCEND_RT_VISIBLE_DEVICES']
print({'model_id': MODEL_ID, 'model_path': MODEL_REF, 'train_data': str(TRAIN_DATA), 'work_dir': str(WORK_DIR), 'npu_count': npu_count, 'visible_npus': VISIBLE_NPUS})

def run_streaming(command: list[str], log_path: Path, env: dict[str, str]) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('将执行：\n' + shlex.join(command))
    with log_path.open('w', encoding='utf-8') as stream:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            stream.write(line)
        returncode = process.wait()
    require(returncode == 0, f'命令失败（退出码 {returncode}）。请检查：{log_path}')
    print('命令输出已保存到：', log_path)


In [ ]:
# 这些参数是本节观察 LoRA 训练过程的起点。
LORA_RANK = 8
LORA_ALPHA = 32
LEARNING_RATE = 1e-4
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
MAX_LENGTH = 512
print('环境、模型和本地 JSONL 数据已准备好。')


### 2. LoRA 参数

| 参数 | 作用 | 当前值 |
| --- | --- | --- |
| `lora_rank` | 低秩矩阵的维度 `r`，决定适配器容量 | 8 |
| `lora_alpha` | 低秩更新的缩放系数 | 32 |
| `learning_rate` | 优化器更新 `A`、`B` 的步长 | `1e-4` |
| `target_modules` | 插入适配器的线性层 | `all-linear` |
| `bfloat16` | 训练时使用的浮点格式 | 开启 |

`lora_rank` 和 `lora_alpha` 决定 LoRA 分支的结构和缩放，`learning_rate` 决定训练过程中每一步走多远。不要把三者混为一谈。

## 3. 检查 SFT 数据

SFT 的一条样本由多轮消息组成。`system` 描述角色，`user` 提出任务，`assistant` 提供监督答案。下面的代码检查 `messages` 是否为非空列表，以及首条消息是否包含 `role` 和 `content`。

In [ ]:
def read_first_record(path: Path) -> dict:
    raw = path.read_text(encoding='utf-8').lstrip()
    require(raw, f'数据文件为空：{path}')
    if raw.startswith('['):
        records = json.loads(raw)
        require(isinstance(records, list) and records and isinstance(records[0], dict), 'JSON 数组的第一项必须是对象。')
        return records[0]
    return json.loads(next(line for line in raw.splitlines() if line.strip()))

first_record = read_first_record(TRAIN_DATA)
require(isinstance(first_record, dict), '训练数据的首条记录必须是对象。')
keys = sorted(first_record.keys())
accepted = {'messages', 'conversations', 'instruction', 'query', 'response'}
require(set(keys) & accepted, f'未识别到常用指令数据字段，当前仅看到：{keys}')
if 'messages' in first_record:
    require(isinstance(first_record['messages'], list) and first_record['messages'], 'messages 必须是非空列表。')
    require({'role', 'content'} <= set(first_record['messages'][0]), 'messages 的首项至少需要 role 与 content 字段。')
print('数据结构检查通过：', keys)


### 检查结果

如果输出包含 `messages`，说明当前 JSONL 可以交给 SFT 数据处理器。继续观察下一步生成的训练命令。


## 4. 生成 3-step 训练命令

训练命令使用 `swift sft`。输入是基础模型和 JSONL 数据，训练方式是 LoRA SFT，输出目录是 `L07-02_smoke`。`SMOKE_STEPS=3` 让实验可以快速完成。


In [ ]:
SMOKE_STEPS = 3
SMOKE_DIR = WORK_DIR / 'L07-02_smoke'
command = [
    SWIFT_BIN, 'sft',
    '--model', MODEL_REF,
    '--dataset', str(TRAIN_DATA),
    '--torch_dtype', 'bfloat16',
    '--tuner_type', 'lora',
    '--target_modules', 'all-linear',
    '--lora_rank', str(LORA_RANK),
    '--lora_alpha', str(LORA_ALPHA),
    '--loss_scale', 'ignore_empty_think',
    '--num_train_epochs', '1',
    '--max_steps', str(SMOKE_STEPS),
    '--per_device_train_batch_size', str(PER_DEVICE_BATCH_SIZE),
    '--gradient_accumulation_steps', str(GRADIENT_ACCUMULATION_STEPS),
    '--learning_rate', str(LEARNING_RATE),
    '--max_length', str(MAX_LENGTH),
    '--logging_steps', '1',
    '--save_steps', str(SMOKE_STEPS),
    '--save_total_limit', '1',
    '--output_dir', str(SMOKE_DIR),
]
env = os.environ.copy()
env['ASCEND_RT_VISIBLE_DEVICES'] = VISIBLE_NPUS
env['NPROC_PER_NODE'] = '1'
env['PYTHONUNBUFFERED'] = '1'


### 命令参数解读

- `--tuner_type lora`：冻结基础模型，训练低秩适配器。
- `--target_modules all-linear`：将 LoRA 注入模型中的线性层。
- `--lora_rank 8`、`--lora_alpha 32`：设置低秩分支的维度和缩放。
- `--max_steps 3`：训练 3 个 step。
- `--logging_steps 1`、`--save_steps 3`：每一步记录 loss，在第 3 步保存 checkpoint。

`bfloat16` 是本次训练使用的浮点格式。

## 5. 执行 smoke test

运行后查看控制台中的 `global_step/max_steps`、`loss` 和进程退出信息。完整训练输出会写入 `notebook_stdout.log`。

In [ ]:
log_path = SMOKE_DIR / 'notebook_stdout.log'
run_streaming(command, log_path, env)


## 6. 读取训练输出

`logging.jsonl` 保存每个训练 step 的日志，checkpoint 保存 LoRA 适配器和训练状态。下面的代码读取 loss 记录并定位最新 checkpoint。


In [ ]:
logging_files = sorted(SMOKE_DIR.rglob('logging.jsonl'), key=lambda path: path.stat().st_mtime)
require(logging_files, f'没有在 {SMOKE_DIR} 找到 logging.jsonl。请检查命令输出：{log_path}')
logging_file = logging_files[-1]
run_dir = logging_file.parent
loss_records = []
for raw in logging_file.read_text(encoding='utf-8').splitlines():
    try:
        record = json.loads(raw)
    except json.JSONDecodeError:
        continue
    loss = record.get('loss', record.get('train_loss'))
    if isinstance(loss, (int, float)) and math.isfinite(float(loss)):
        loss_records.append(record)
require(loss_records, f'{logging_file} 中没有有限的 loss 记录。')
checkpoints = [path for path in run_dir.glob('checkpoint-*') if path.is_dir()]
require(checkpoints, f'没有找到 checkpoint；请检查 {run_dir} 的训练输出。')
print('运行目录：', run_dir)
print('logging.jsonl：', logging_file)
print('有限 loss 记录数：', len(loss_records))
print('checkpoint：', checkpoints[-1])
print('smoke test 通过：训练命令返回 0，日志含有限 loss，且已保存 checkpoint。')


## 7. 小结与思考

1. 在这条命令中，哪些参数决定 LoRA 适配器的大小？
2. `logging_steps=1` 对日志曲线有什么影响？
3. 你在输出中看到了哪些文件？它们分别对应数据、日志和适配器的哪一部分？
4. 进入 L07-03，把 `max_steps` 改为 50，观察更完整的 loss 轨迹。
